In [ ]:
import numpy as np
from numpy import deg2rad as d2r
from numpy import array as arr

from scipy.optimize import minimize, approx_fprime

from space_traj_opt.math.orbital_calcs import rv_to_aei
from space_traj_opt.models.controller3d import CtrlMode, flight_path_angle, lts_control
from space_traj_opt.optimization.phases import DynEnum, Phase, TerminalConditions
from space_traj_opt.optimization.constraints import ConstraintType

from space_traj_opt.optimization.problem import Problem
from space_traj_opt.optimization.transcription import MultiShootingTranscription
from space_traj_opt.optimization.utils import  normalize_decision_vec, traj_rollout, denormalize_decision_vec
from space_traj_opt.postprocessing.jac_viz import plot, visualize_jac2
from space_traj_opt.postprocessing.utils import unpack_sol_list


STANDARD_GRAV = 9.80665


## Electron Rocket Parameters

In [ ]:
n_engines_s1 = 9
n_engines_s2 = 1
isp_s1 = 311.0
engine_thrust_s1 = n_engines_s1*24910.04  # N Average between sl and vac
isp_s2 = 343.0
engine_thrust_s2 = n_engines_s2* 25_000.0  # N
s1_vch_params = (engine_thrust_s1, isp_s1, True)
s2_vch_params = (engine_thrust_s2, isp_s2, False)

fairing_mass = 50.0
farinig_timing = 184.0 - 162.0 # sec
payload = 250.0
s1_dry_mass = 1076.47308279  
s2_dry_mass = 257.90093739  

s1_wet_mass = 10047.082106064723
s2_wet_mass = 2602.454913676189
total_mass = 12949.537019740912

mdot_s1 = engine_thrust_s1 / STANDARD_GRAV / isp_s1
mdot_s2 = engine_thrust_s2 / STANDARD_GRAV / isp_s2

In [ ]:
NUM_X= 7
NUM_U = 4
NUM_PHASE = 1
# %load_ext snakeviz

## Initial Guesses

In [ ]:
mu_earth = 3.986004418e14
earth_r = 6_378_000.0 # m
circ_orbit_alt = 200_000.0 
v_circ = np.sqrt(mu_earth / (earth_r + circ_orbit_alt))

# Guesses 
s2_sep_mass = s2_wet_mass + payload + fairing_mass
x0 = arr([
    [0, 84918 + earth_r,0, 2457, 997, 0, s2_wet_mass + payload - farinig_timing * mdot_s2]
])


## normalization vector 
x0_n_vec = arr([earth_r, earth_r, earth_r, 5000, 1000, 1000, 5000])

e_desired = 0.0
a_desired = earth_r + circ_orbit_alt

In [ ]:
x0

## Define a multiphase trajectory problem

In [ ]:
phase = Phase(
    name="phase0",
    dynamics_type=DynEnum.DYNAMICS_3D,
    control_type=CtrlMode.LTS,
    model_params=s2_vch_params,
)


## Define state, control and time guesses for each phase 

In [ ]:

phase.set_state( x0[0], bounds=x0[0], norm_vec=x0_n_vec)
phase.set_controller(
    CtrlMode.LTS,
    u0=arr([-0.0013, 0.62, 0.0, 0.0]),
    bounds=[(-0.1, 0.1), (-3, 3), (-0.1, 0.1), (-3, 3)],
    norm_vec=[0.1, np.pi / 2, 0.1, np.pi / 2],
)
phase.set_time(300.0)

problem_builder = MultiShootingTranscription(["phase0"], num_states=7)
problem_builder.add_phase("phase0", phase)

# terminal condition for the target orbit
x_f = arr([a_desired, e_desired, v_circ, s2_dry_mass + payload])
xf_n_vec = arr([a_desired, 1.0, v_circ, 5000])
problem_builder.add_terminal(
    TerminalConditions.set_terminal(
        x_final=x_f,
        bounds=[a_desired, e_desired, v_circ, None],
        norm_vec=xf_n_vec
    )
)


## Build The problem
Builds the decision vector and bounds 

In [ ]:
d0, d_bounds, normalization_vec, full_params = problem_builder.build()
d0_norm, d_bounds_norm = normalize_decision_vec(d0, d_bounds, normalization_vec)

problem = Problem(
    d0_norm,
    d_bounds_norm,
    normalization_vec,
    num_states=7,
    num_phases=1,
    num_terminal_states=4,
    terminal_con_kind=ConstraintType.ON_ORBIT
)


In [ ]:
config = full_params[0]
u, x, t_terminal, control_law = problem.unpack_decision_var(d0, config=config)


In [ ]:
u_ = tuple(u.tolist())
x_ = tuple(x.tolist())
t_ = float(t_terminal)
vch_params = (DynEnum.DYNAMICS_3D, config[-1],  (control_law, u_))



In [ ]:
sol = traj_rollout(t_, x_, vch_params)


In [ ]:
plot(
    sol.t, sol.y[1]-earth_r,
    title="Time vs States", 
    xlabel="Time", 
    ylabel="Pos y",
    )

## Defining dynamic constraint function

In [ ]:

def dynamics_knot_constrant(decision_var, config_list): 
    """Integrate the dynamics of each segment. Calcculate the defect  between the knot points.
    This vector is used as the equality constraint for the optimization problem.
    The defect is calculated as the difference between the final state of the previous segment and the initial state of the next segment.

    Args:
        decision_var : Optimzation decission vector
        config_list : List of configs for each phase

    Returns:
        Knot defect vector
    """
    d0 = denormalize_decision_vec(decision_var, normalization_vec)
    defect_vector_list = []
    sol_list=  problem.full_traj_rollout(d0, config_list)
    for idx in range(1,NUM_PHASE):
        _,_, knot_defect,_,_ = config_list[idx]
        defect_sub_vector = sol_list[idx].y[:,0] - sol_list[idx-1].y[:,-1] + knot_defect
        defect_sub_vector /= arr([10000, 10000, 10000, 8000, 5000, 5000, 1000])# defect vector normalization
        defect_vector_list.append(defect_sub_vector)
    
    # Terminal Defect
    # calculate orbital elements here
    m_desired = d0[-1]
    e_scale = 1.0

    r = sol_list[-1].y[:,-1][:3]
    v = sol_list[-1].y[:,-1][3:6]
    m = sol_list[-1].y[:,-1][6]
    a, e, i = rv_to_aei(r, v, mu_earth)
    v_mag = np.linalg.norm(v)
    terminal_defect = np.array([
        (a   - a_desired) / earth_r,
        (e   - e_desired) / e_scale,
        (v_mag   - v_circ) / 1000,  # Normalize velocity
        (m   - m_desired) / 100,
        #(i   - i_desired) / 100,

    ])
    defect_vector_list.append(terminal_defect)
    defect_vec = arr(defect_vector_list).flatten()
    return defect_vec



In [ ]:
dynamics_knot_constrant(d0_norm, full_params)

In [ ]:
constraints = [{'type': 'eq', 'fun': dynamics_knot_constrant, 'args':(full_params,) },]

## Objective function
Maximize stage 2 mass

In [ ]:
def objective(decision_var: tuple, params: tuple) -> float:
    """Objective function for min prop

    Args:
        decision_var : Optimization problem decision vector
        params : 

    Returns:
        Cost to minimize
    """
    terminal_mass= decision_var[-1]
    return -terminal_mass*terminal_mass*10000


def jac_objective(decision_var: tuple, params: tuple):
    """Jac of the decision vector wrt the cost."""
    
    jac = np.zeros_like(decision_var)
    val = -decision_var[-1] - decision_var[-1]
    jac[-1]= val*10000
    return jac

## Scipy Minimize
SLSQP has to be used here because it can handle bounds and equality constraints.

In [ ]:
# %%snakeviz

result = minimize(
    objective,  
    d0_norm, 
    jac= jac_objective,
    method='SLSQP', 
    bounds=d_bounds_norm, 
    constraints=constraints,
    options = {"maxiter": 500, "disp": True},
    args=(full_params,)
)

In [ ]:
constraint_jac = approx_fprime(
    result.x, 
    dynamics_knot_constrant, 
    np.float64(1.4901161193847656e-08), full_params)

In [ ]:
visualize_jac2(result.x, constraint_jac)

In [ ]:
x_opt = denormalize_decision_vec(result.x, normalization_vec)

sol_list = problem.full_traj_rollout(x_opt, full_params)

In [ ]:
sol_list[0]

In [ ]:
## Post processing

state = sol_list[0].y
t_sol = sol_list[0].t

ctr_param, x0, time, ctrl_mode = problem.unpack_decision_var(d0, full_params[0] )

pos = state[0:3]
vel = state[3:6]
mass = state[6]
a_values = []
e_values = []
i_values = []
fpa_values = []
lts_values = []
pitch_values = []
for p, v in zip(pos.T, vel.T):
    a, e, i = rv_to_aei(p, v, mu_earth)
    a_values.append(a)
    e_values.append(e)
    i_values.append(i)
    fpa_values.append(flight_path_angle(p, v))

for t, cur_x in zip(t_sol, state.T):
    lts_params = lts_control(t, cur_x,ctr_param )
    lts_values.append(lts_params)
    pitch_values.append(np.atan2(lts_params[1], lts_params[0]) )

a = np.array(a_values)
e = np.array(e_values)
i = np.array(i_values)
fpa = np.array(fpa_values)
pitch = np.array(pitch_values)
v_eci_mag = np.linalg.norm(vel, axis=0) 
r_periapsis = a * (1.0 - e)
r_apoapsis  = a * (1.0 + e)

h_periapsis = r_periapsis - earth_r
h_apoapsis  = r_apoapsis  - earth_r



In [ ]:
plot(
    t_sol, i,
    title="Time vs States", 
    xlabel="Time", 
    ylabel="Inclination",
    )

In [ ]:
plot(
    t_sol, [fpa, pitch],
    title="Time vs States", 
    xlabel="Time", 
    ylabel="Flight Path Angle, Pitch_RSW",
    )

In [ ]:
plot(
    t_sol, [h_periapsis, h_apoapsis],
    title="Time vs States", 
    xlabel="Time", 
    ylabel="perigee and apogee altitudes",
    )

In [ ]:
plot(
    t_sol, e,
    title="Time vs States", 
    xlabel="Time", 
    ylabel="Eccentricity",
    )

In [ ]:
plot(
    t_sol, mass,
    title="Time vs States", 
    xlabel="Time", 
    ylabel="Mass",
    )

In [ ]:
plot(
    t_sol, v_eci_mag,
    title="Time vs States", 
    xlabel="Time", 
    ylabel="vel mag",
    )

In [ ]:
plot(
    *unpack_sol_list(sol_list,0),
    title="Time vs States", 
    xlabel="Time", 
    ylabel="Pos x",
    trace_names=("phase0", "phase1", "phase2", "phase3")
    )